In [1]:
"""
Taller - Modelos de Reflexión: Lambert, Phong, Blinn-Phong y PBR
Entorno: Google Colab (matplotlib + ipywidgets + numpy)

Cómo usar:
  1. Pega este código en una celda de Google Colab y ejecútala.
  2. Usa los sliders para cambiar parámetros en tiempo real.
  3. Haz clic en "Renderizar" para actualizar la imagen.

NO requiere pygame ni ventana gráfica.
"""

# ── Instalar dependencias (solo la primera vez) ───────────────
# En Colab numpy y matplotlib ya vienen instalados.
# ipywidgets también, pero hay que activar la extensión:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"], check=False)

import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output

# ─────────────────────────────────────────────────────────────
# PARÁMETROS DE ESCENA
# ─────────────────────────────────────────────────────────────
SPHERE_R = 120          # radio en píxeles
N_SPHERES = 4
AMBIENT   = 0.06
ALBEDO    = np.array([0.22, 0.50, 0.85], dtype=np.float64)

# Posición fija de la cámara (Z grande = "lejos")
CAM_Z = 600.0

# ─────────────────────────────────────────────────────────────
# PRE-COMPUTAR GEOMETRÍA DE LA ESFERA (una sola vez)
# ─────────────────────────────────────────────────────────────
def build_sphere(r):
    """
    Devuelve máscara booleana y coordenadas locales 3D
    normalizadas de cada píxel dentro de la esfera.
    Shape de todo: (2r+1, 2r+1)
    """
    size = 2 * r + 1
    # y_idx, x_idx: índices de fila y columna
    y_idx, x_idx = np.mgrid[0:size, 0:size]
    lx = (x_idx - r).astype(np.float64)
    ly = (y_idx - r).astype(np.float64)
    d2 = lx**2 + ly**2
    mask = d2 <= float(r * r)
    lz   = np.where(mask, np.sqrt(np.maximum(float(r*r) - d2, 0.0)), 0.0)
    return mask, lx, ly, lz

mask, LX, LY, LZ = build_sphere(SPHERE_R)
SIZE = 2 * SPHERE_R + 1   # tamaño del bounding box


# ─────────────────────────────────────────────────────────────
# CÁLCULO DE VECTORES N, L, V
# ─────────────────────────────────────────────────────────────
def compute_vectors(lx, ly, lz, mask, light_pos_3d):
    """
    Calcula los vectores normalizados N (normal),
    L (hacia la luz) y V (hacia la cámara) para
    cada píxel de la esfera.

    light_pos_3d: (x, y, z) en coordenadas de escena.
      x, y en [-W/2, W/2]  y  [-H/2, H/2]
      z positivo = delante de la esfera
    """
    # ── Normal ────────────────────────────────────────────────
    # Para una esfera perfecta, la normal en cada punto
    # es simplemente el vector desde el centro hasta el punto.
    # Como lx, ly, lz ya son coordenadas locales al centro,
    # solo hay que normalizar (dividir por su módulo).
    length = np.sqrt(lx**2 + ly**2 + lz**2 + 1e-10)
    Nx =  lx / length    # componente X de la normal
    Ny = -ly / length    # Y invertido: pantalla Y crece hacia abajo, mundo Y hacia arriba
    Nz =  lz / length

    # ── Posición 3D del punto en la escena ───────────────────
    # El centro de la esfera está en el origen (0,0,0).
    # Px, Py son las coordenadas mundiales del píxel.
    Px = lx           # X local = X mundial (esfera centrada)
    Py = -ly          # invertir Y para pasar a coordenadas mundiales
    Pz = lz           # Z del punto en la esfera

    # ── Vector L: del punto hacia la luz ──────────────────────
    Lx = light_pos_3d[0] - Px
    Ly = light_pos_3d[1] - Py
    Lz = light_pos_3d[2] - Pz
    Ll = np.sqrt(Lx**2 + Ly**2 + Lz**2 + 1e-10)
    Lx /= Ll;  Ly /= Ll;  Lz /= Ll

    # ── Vector V: del punto hacia la cámara ──────────────────
    # La cámara está en (0, 0, CAM_Z)
    Vx = -Px
    Vy = -Py
    Vz = CAM_Z - Pz
    Vl = np.sqrt(Vx**2 + Vy**2 + Vz**2 + 1e-10)
    Vx /= Vl;  Vy /= Vl;  Vz /= Vl

    return (Nx, Ny, Nz), (Lx, Ly, Lz), (Vx, Vy, Vz)


def dot3(Ax, Ay, Az, Bx, By, Bz):
    """Producto punto entre dos vectores expresados por componentes."""
    return Ax*Bx + Ay*By + Az*Bz


# ─────────────────────────────────────────────────────────────
# MODELOS DE ILUMINACIÓN
# ─────────────────────────────────────────────────────────────

def shade_lambert(N, L, V, shininess, roughness, metalness):
    """
    Modelo Lambert (difuso puro).

    Ecuación:
        I_difusa = kd * max(N · L, 0)

    kd  = albedo del material (color)
    N·L = coseno del ángulo entre la normal y la luz.
          Si la luz está perpendicular a la superficie → 1.0 (máximo brillo).
          Si la luz es rasante → 0.0 (sin brillo).
    """
    Nx, Ny, Nz = N
    Lx, Ly, Lz = L

    NdotL = np.maximum(dot3(Nx, Ny, Nz, Lx, Ly, Lz), 0.0)

    # Color final = albedo × (ambiente + componente difusa)
    r = ALBEDO[0] * (AMBIENT + NdotL)
    g = ALBEDO[1] * (AMBIENT + NdotL)
    b = ALBEDO[2] * (AMBIENT + NdotL)
    return r, g, b


def shade_phong(N, L, V, shininess, roughness, metalness):
    """
    Modelo Phong (difuso + especular).

    El especular usa el vector reflejado R:
        R = 2*(N·L)*N - L
        I_especular = ks * max(R · V, 0) ^ shininess

    R·V = cuánto se alinea el ojo con la dirección de rebote.
    shininess = concentración del brillo:
        bajo  (1-10)  → brillo difuso, como plástico mate
        medio (32-64) → brillo típico
        alto  (256+)  → brillo muy concentrado, como metal pulido
    """
    Nx, Ny, Nz = N
    Lx, Ly, Lz = L
    Vx, Vy, Vz = V

    NdotL = np.maximum(dot3(Nx, Ny, Nz, Lx, Ly, Lz), 0.0)

    # Vector reflejado: R = 2*(N·L)*N - L
    # Esto da la dirección exacta en la que "rebota" el rayo de luz
    Rx = 2 * NdotL * Nx - Lx
    Ry = 2 * NdotL * Ny - Ly
    Rz = 2 * NdotL * Nz - Lz
    Rl = np.sqrt(Rx**2 + Ry**2 + Rz**2 + 1e-10)
    Rx /= Rl;  Ry /= Rl;  Rz /= Rl

    RdotV = np.maximum(dot3(Rx, Ry, Rz, Vx, Vy, Vz), 0.0)
    spec  = np.power(RdotV, shininess)

    r = ALBEDO[0] * (AMBIENT + NdotL) + 0.85 * spec
    g = ALBEDO[1] * (AMBIENT + NdotL) + 0.85 * spec
    b = ALBEDO[2] * (AMBIENT + NdotL) + 0.85 * spec
    return r, g, b


def shade_blinnphong(N, L, V, shininess, roughness, metalness):
    """
    Modelo Blinn-Phong (difuso + especular con half vector).

    Optimización de Phong: en lugar de calcular R exacto,
    se usa el half vector H = normalize(L + V).
    Luego se mide N·H en vez de R·V.

    Ventajas sobre Phong:
    - Más barato de calcular (no hay que calcular reflect())
    - Se comporta mejor en ángulos rasantes
    - Por eso fue el estándar en GPU por décadas (OpenGL clásico)
    """
    Nx, Ny, Nz = N
    Lx, Ly, Lz = L
    Vx, Vy, Vz = V

    NdotL = np.maximum(dot3(Nx, Ny, Nz, Lx, Ly, Lz), 0.0)

    # Half vector: a mitad de camino entre L y V
    Hx = Lx + Vx
    Hy = Ly + Vy
    Hz = Lz + Vz
    Hl = np.sqrt(Hx**2 + Hy**2 + Hz**2 + 1e-10)
    Hx /= Hl;  Hy /= Hl;  Hz /= Hl

    # Especular: N·H (cuánto H se alinea con la normal)
    NdotH = np.maximum(dot3(Nx, Ny, Nz, Hx, Hy, Hz), 0.0)
    spec  = np.power(NdotH, shininess)

    r = ALBEDO[0] * (AMBIENT + NdotL) + 0.85 * spec
    g = ALBEDO[1] * (AMBIENT + NdotL) + 0.85 * spec
    b = ALBEDO[2] * (AMBIENT + NdotL) + 0.85 * spec
    return r, g, b


def shade_pbr(N, L, V, shininess, roughness, metalness):
    """
    PBR: Cook-Torrance BRDF.

    En lugar de 'inventar' parámetros visuales como shininess,
    se usan parámetros físicos reales:

      roughness  (rugosidad):  0 = espejo perfecto, 1 = completamente mate
      metalness  (metalicidad): 0 = plástico/dieléctrico, 1 = metal puro

    La fórmula tiene 3 componentes para el especular:
      D = GGX  : distribución de microfacetas (cuántas "mini-espejos"
                  apuntan hacia H). Con roughness bajo, solo las que
                  apuntan exactamente a H contribuyen → brillo concentrado.

      F = Fresnel (Schlick): a ángulos rasantes, TODOS los materiales
                  se vuelven más reflectivos (piensa en el agua a contraluz).
                  F0 es la reflectividad a 0°:
                    dieléctrico → F0 ≈ 0.04
                    metal       → F0 = albedo (el metal colorea su especular)

      G = Smith  : auto-oclusión de microfacetas. Superficies muy rugosas
                  tienen más zonas en sombra entre microfacetas.
    """
    Nx, Ny, Nz = N
    Lx, Ly, Lz = L
    Vx, Vy, Vz = V

    NdotL = np.maximum(dot3(Nx, Ny, Nz, Lx, Ly, Lz), 0.0)
    NdotV = np.maximum(dot3(Nx, Ny, Nz, Vx, Vy, Vz), 1e-4)

    # Half vector
    Hx = Lx + Vx;  Hy = Ly + Vy;  Hz = Lz + Vz
    Hl = np.sqrt(Hx**2 + Hy**2 + Hz**2 + 1e-10)
    Hx /= Hl;  Hy /= Hl;  Hz /= Hl

    NdotH = np.maximum(dot3(Nx, Ny, Nz, Hx, Hy, Hz), 0.0)
    VdotH = np.maximum(dot3(Vx, Vy, Vz, Hx, Hy, Hz), 0.0)

    a  = roughness * roughness      # alpha (convención física)
    a2 = a * a

    # ── D: distribución GGX ──────────────────────────────────
    # Cuántas microfacetas están orientadas hacia H
    denom_D = NdotH**2 * (a2 - 1.0) + 1.0
    D = a2 / (math.pi * denom_D**2 + 1e-6)

    # ── F: Fresnel Schlick ────────────────────────────────────
    # F0: reflectividad base. Metales usan su albedo como F0.
    F0 = np.array([0.04, 0.04, 0.04]) * (1 - metalness) + ALBEDO * metalness
    fres = (1.0 - VdotH) ** 5
    Fr = F0[0] + (1 - F0[0]) * fres
    Fg = F0[1] + (1 - F0[1]) * fres
    Fb = F0[2] + (1 - F0[2]) * fres

    # ── G: geometría Smith ────────────────────────────────────
    # Auto-oclusión de microfacetas
    k   = (roughness + 1)**2 / 8.0
    G1v = NdotV / (NdotV * (1 - k) + k)
    G1l = np.maximum(NdotL, 1e-8) / (np.maximum(NdotL, 1e-8) * (1 - k) + k)
    G   = G1v * G1l

    # ── Especular Cook-Torrance ───────────────────────────────
    dSpec = 4.0 * NdotV * NdotL + 1e-6
    sr = D * Fr * G / dSpec
    sg = D * Fg * G / dSpec
    sb = D * Fb * G / dSpec

    # ── Difuso: los metales no tienen componente difusa ───────
    # kd = (1 - F) * (1 - metalness)
    kdr = (1 - Fr) * (1 - metalness)
    kdg = (1 - Fg) * (1 - metalness)
    kdb = (1 - Fb) * (1 - metalness)

    dr = kdr * ALBEDO[0] / math.pi
    dg = kdg * ALBEDO[1] / math.pi
    db = kdb * ALBEDO[2] / math.pi

    # ── Color final + tone mapping Reinhard ──────────────────
    # Reinhard evita que los valores superen 1.0 (overexposure)
    # fórmula: x_out = x / (x + 1)
    def reinhard(x): return x / (x + 1.0)

    r = reinhard(AMBIENT * ALBEDO[0] + (dr + sr) * NdotL)
    g = reinhard(AMBIENT * ALBEDO[1] + (dg + sg) * NdotL)
    b = reinhard(AMBIENT * ALBEDO[2] + (db + sb) * NdotL)
    return r, g, b


SHADERS = [shade_lambert, shade_phong, shade_blinnphong, shade_pbr]
NAMES   = ["Lambert", "Phong", "Blinn-Phong", "PBR"]


# ─────────────────────────────────────────────────────────────
# RENDERIZADO
# ─────────────────────────────────────────────────────────────
def render_sphere(shader_fn, light_pos_3d, shininess, roughness, metalness):
    """
    Renderiza UNA esfera con el shader dado.
    Devuelve un array RGB (SIZE, SIZE, 3) uint8.
    """
    N_vec, L_vec, V_vec = compute_vectors(LX, LY, LZ, mask, light_pos_3d)

    cr, cg, cb = shader_fn(N_vec, L_vec, V_vec, shininess, roughness, metalness)

    img = np.zeros((SIZE, SIZE, 3), dtype=np.float64)
    img[:, :, 0] = np.where(mask, cr, 0.05)
    img[:, :, 1] = np.where(mask, cg, 0.05)
    img[:, :, 2] = np.where(mask, cb, 0.05)

    return np.clip(img, 0, 1)


def render_scene(light_x, light_y, light_z, shininess, roughness, metalness):
    """
    Renderiza las 4 esferas en un solo canvas matplotlib.
    """
    light_pos = np.array([light_x, light_y, float(light_z)], dtype=np.float64)

    PAD    = 20          # píxeles de separación entre esferas
    canvas_w = N_SPHERES * SIZE + (N_SPHERES - 1) * PAD
    canvas_h = SIZE + 80  # espacio extra abajo para etiquetas
    canvas   = np.full((canvas_h, canvas_w, 3), 0.07)

    for i, shader in enumerate(SHADERS):
        img = render_sphere(shader, light_pos, shininess, roughness, metalness)
        x0  = i * (SIZE + PAD)
        canvas[:SIZE, x0:x0+SIZE] = img

    fig, ax = plt.subplots(figsize=(14, 4.5))
    ax.imshow(canvas, interpolation='bilinear')
    ax.axis('off')

    # Etiquetas de modelos
    for i, name in enumerate(NAMES):
        cx = i * (SIZE + PAD) + SIZE // 2
        ax.text(cx, SIZE + 18, name,
                ha='center', va='top', fontsize=13,
                color='#cccccc', fontweight='bold')

    # Parámetros activos
    params = (f"shininess={shininess:.0f}   roughness={roughness:.2f}   "
              f"metalness={metalness:.2f}   luz=({light_x:.0f}, {light_y:.0f}, {light_z:.0f})")
    ax.text(canvas_w // 2, SIZE + 50, params,
            ha='center', va='top', fontsize=10, color='#888888')

    # Indicador de posición de la luz (sobre la primera esfera como referencia)
    ax.set_title("Modelos de Reflexión de Luz — renderizado por píxel",
                 fontsize=13, color='#aaaaaa', pad=8)

    plt.tight_layout(pad=0.5)
    plt.show()


# ─────────────────────────────────────────────────────────────
# INTERFAZ IPYWIDGETS
# ─────────────────────────────────────────────────────────────

# Sliders
w_lx = widgets.FloatSlider(
    value=80, min=-SPHERE_R, max=SPHERE_R, step=5,
    description='Luz X:', continuous_update=False,
    style={'description_width': '80px'}, layout=widgets.Layout(width='450px'))

w_ly = widgets.FloatSlider(
    value=100, min=-SPHERE_R, max=SPHERE_R, step=5,
    description='Luz Y:', continuous_update=False,
    style={'description_width': '80px'}, layout=widgets.Layout(width='450px'))

w_lz = widgets.FloatSlider(
    value=300, min=50, max=600, step=10,
    description='Luz Z:', continuous_update=False,
    style={'description_width': '80px'}, layout=widgets.Layout(width='450px'))

w_shine = widgets.FloatLogSlider(
    value=64, base=2, min=0, max=9, step=0.5,
    description='Shininess:', continuous_update=False,
    style={'description_width': '80px'}, layout=widgets.Layout(width='450px'))

w_rough = widgets.FloatSlider(
    value=0.4, min=0.0, max=1.0, step=0.05,
    description='Roughness:', continuous_update=False,
    style={'description_width': '80px'}, layout=widgets.Layout(width='450px'))

w_metal = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.1,
    description='Metalness:', continuous_update=False,
    style={'description_width': '80px'}, layout=widgets.Layout(width='450px'))

btn = widgets.Button(
    description='⟳  Renderizar',
    button_style='primary',
    layout=widgets.Layout(width='180px', height='36px'))

output = widgets.Output()

def on_render(_):
    with output:
        clear_output(wait=True)
        render_scene(
            w_lx.value, w_ly.value, w_lz.value,
            w_shine.value, w_rough.value, w_metal.value
        )

btn.on_click(on_render)

# Layout de controles
col1 = widgets.VBox([w_lx, w_ly, w_lz])
col2 = widgets.VBox([w_shine, w_rough, w_metal])

panel = widgets.VBox([
    widgets.HTML("<h3 style='color:#aaa;margin:8px 0 4px'>Modelos de Reflexión de Luz</h3>"
                 "<p style='color:#777;font-size:12px;margin:0 0 8px'>"
                 "Ajusta los parámetros y haz clic en Renderizar.</p>"),
    widgets.HBox([col1, widgets.HTML("&nbsp;&nbsp;&nbsp;"), col2]),
    widgets.HTML("<p style='color:#666;font-size:11px;margin:4px 0 2px'>"
                 "<b>Shininess</b>: afecta Phong y Blinn-Phong. "
                 "<b>Roughness/Metalness</b>: solo afectan PBR.</p>"),
    btn,
    output
])

display(panel)

# Render inicial automático
on_render(None)